# Lab 02 · Mesmos dados. Três algoritmos
**Machine Learning na Prática · Dia 1 · Módulo 03: KNN, Árvores & Random Forest**

Os dados já chegam **prontos**: o clássico **Iris** (150 flores, 4 medidas, 3 espécies), sem faltantes nem sujeira.
O foco é ver os três modelos funcionando e escolher um com honestidade.

| Etapa | O que fazer |
|---|---|
| 1. Olhar e separar | explorar o Iris e separar treino/teste com estratificação |
| 2. Comparar | KNN × Árvore × Random Forest com validação cruzada no treino |
| 3. Ajustar | variar K e profundidade; desenhar a árvore |
| 4. Avaliar | o escolhido no teste: matriz de confusão e métricas |

Procure as células marcadas com **`# TODO`** e substitua os `___`.

> **Gabarito.** Versão do professor, com todas as células resolvidas. Não compartilhe antes da atividade.

### Bibliotecas
No **Google Colab** tudo já vem instalado: esta célula só confere. Em outro ambiente (Jupyter no seu PC), ela instala o que faltar.

In [ ]:
import importlib.util, subprocess, sys
pacotes = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'sklearn': 'scikit-learn'}
faltando = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltando])
print('Bibliotecas prontas!' if not faltando else f'Instalado agora: {", ".join(faltando)}')

## 0 · Preparação (célula pronta)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
nomes = iris.target_names
X.head()

> 🔎 **Gabarito · o que observar**
>
> - 4 medidas em centímetros por flor, todas na mesma ordem de grandeza.
> - O alvo `target` vale 0, 1 ou 2 (setosa, versicolor, virginica).

## 1 · Olhar e separar

In [ ]:
# TODO: quantas flores de cada espécie?
print(y.value_counts())

plt.scatter(X['petal length (cm)'], X['petal width (cm)'], c=y, cmap='viridis', alpha=.8)
plt.xlabel('comprimento da pétala'); plt.ylabel('largura da pétala'); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - **50 flores de cada espécie**: classes perfeitamente balanceadas, então a acurácia é uma métrica honesta aqui.
> - No gráfico de pétalas, a **setosa** forma um grupo isolado (pétalas pequenas). **Versicolor e virginica** se encostam: é ali que os modelos vão errar.

**Estratificar** mantém a mesma proporção de espécies no treino e no teste — importante quando há poucas amostras.

In [ ]:
# TODO: 25% para teste, estratificando pelo alvo
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(y_tr.value_counts().to_dict(), y_te.value_counts().to_dict())

> 🔎 **Gabarito · o que observar**
>
> - Treino: 38 / 37 / 37. Teste: 12 / 13 / 13.
> - As proporções se mantêm (≈ 1/3 de cada). Sem estratificar, o teste de 38 flores poderia, por azar, ficar com pouquíssimas de uma espécie.

## 2 · KNN × Árvore × Random Forest
- **KNN** guarda os exemplos e decide pela votação dos vizinhos. Usa **distâncias**, então vai num `Pipeline` com padronização.
- **Árvore** aprende perguntas do tipo “pétala ≤ 2,45?”. Não liga para escala.
- **Random Forest** treina muitas árvores, cada uma com um sorteio diferente, e decide pelo voto.

Os três têm a mesma interface (`fit`, `predict`), então dá para comparar todos num único `for`, pela **validação cruzada no treino**.

In [ ]:
modelos = {
    # TODO: KNN com padronização dentro do pipeline
    'KNN (K=5)': Pipeline([('padronizar', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))]),
    'Árvore': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

for nome, modelo in modelos.items():
    # TODO: acurácia da validação cruzada com 5 folds, só no treino
    acc = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='accuracy')
    print(f'{nome:<14} {acc.mean():.3f} ± {acc.std():.3f}')

> 🔎 **Gabarito · o que observar**
>
> - KNN **0,946 ± 0,034**; árvore **0,929 ± 0,044**; random forest **0,946 ± 0,034**.
> - As diferenças são **menores que o desvio** entre folds: estatisticamente, os três empatam neste problema.
> - A árvore única é a mais instável (maior desvio), e a floresta reduz essa variância ao combinar 200 árvores.

## 3 · Variar K e profundidade
Compare a acurácia **no treino** com a da **validação cruzada**. Quando as duas se afastam, o modelo está decorando.

In [ ]:
def treino_vs_cv(modelo):
    acc_tr = modelo.fit(X_tr, y_tr).score(X_tr, y_tr)
    acc_cv = cross_val_score(modelo, X_tr, y_tr, cv=5).mean()
    return f'treino {acc_tr:.3f} | validação {acc_cv:.3f}'

for k in [1, 5, 15, 61]:
    # TODO: KNN com K = k
    knn = Pipeline([('padronizar', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=k))])
    print(f'KNN, K = {k:<4}', treino_vs_cv(knn))

for p in [1, 2, 3, None]:
    # TODO: árvore com profundidade máxima = p
    arvore = DecisionTreeClassifier(max_depth=p, random_state=42)
    print(f'Árvore, prof. = {str(p):<4}', treino_vs_cv(arvore))

> 🔎 **Gabarito · o que observar**
>
> **KNN:** K = 1 dá 100% no treino, mas 0,946 na validação cruzada. Entre K = 3 e 15, a validação fica estável em ≈ 0,95. Com K = 31 e 61, as duas curvas **caem juntas** (0,87 e 0,73): o modelo ficou simples demais.
>
> **Árvore:** profundidade 1 = 0,65 (uma pergunta só separa 2 das 3 espécies). Com profundidade 2 ou 3, a validação chega a ≈ 0,93–0,94. De 4 em diante, o treino vai a 99–100% e a validação **não melhora** (0,93).
>
> - **Sobreajuste**: a distância entre treino e validação aumenta. **Subajuste**: as duas curvas ficam baixas.

**Para pensar:** com K = 1 e com a árvore sem limite (`None`), a acurácia no treino é 100%. Isso significa que são os melhores modelos?

Uma árvore rasa é fácil de ler. Veja as perguntas que ela aprendeu:

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** não. 100% no treino com K = 1 é trivial: cada flor é a própria vizinha mais próxima.
> A árvore sem limite cresce até separar cada ponto. O que importa é a validação cruzada, e nela esses modelos não são os melhores.

In [ ]:
arvore = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_tr, y_tr)
plt.figure(figsize=(10, 5))
plot_tree(arvore, feature_names=X.columns, class_names=nomes, filled=True); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - A árvore inteira cabe em **duas perguntas**, e as duas são sobre `petal length (cm)`:
>   - comprimento da pétala ≤ 2,45 → **setosa** (nó puro);
>   - senão, ≤ 4,85 → **versicolor**, e acima disso → **virginica**.
> - As sépalas nem aparecem: a árvore escolheu sozinha a feature mais informativa.
> - Com só 2 níveis, ela acerta 96% do treino. É um modelo que dá para **explicar** a qualquer pessoa.

## 4 · Avaliar o escolhido no teste
Escolha um modelo **pela validação cruzada** (não pelo teste!). Só agora o teste é usado — uma única vez.

In [ ]:
# TODO: escolha o modelo (ex.: modelos['Random Forest']) e treine no treino inteiro
escolhido = modelos['Random Forest'].fit(X_tr, y_tr)
pred = escolhido.predict(X_te)

ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=nomes); plt.show()
# TODO: precision, recall e F1 por classe
print(classification_report(y_te, pred, target_names=nomes))

> 🔎 **Gabarito · o que observar**
>
> Matriz de confusão (linhas = real, colunas = previsto):
>
> | | setosa | versicolor | virginica |
> |---|---|---|---|
> | **setosa** | 12 | 0 | 0 |
> | **versicolor** | 0 | 12 | 1 |
> | **virginica** | 0 | 3 | 10 |
>
> - Acurácia no teste: **0,89** (34 de 38), um pouco abaixo da validação cruzada (0,946). Com só 38 flores, cada erro vale 2,6 pontos percentuais.
> - A setosa nunca erra. **Todos os 4 erros** são entre versicolor e virginica, justamente as espécies que se encostam no gráfico.
> - Virginica tem recall 0,77: 3 das 13 foram chamadas de versicolor. Versicolor tem precision 0,80, porque recebeu esses 3 “intrusos”.

## 5 · Desafios extras (se sobrar tempo)
### 5.1 Classifique uma flor nova
Invente as medidas de duas flores e veja o que o modelo diz — e com quanta certeza.

In [ ]:
# TODO: troque as medidas (comprimento e largura da sépala, comprimento e largura da pétala, em cm)
novas = pd.DataFrame([[5.0, 3.4, 1.5, 0.2],
                      [6.0, 2.9, 4.9, 1.6]], columns=X.columns)
print(nomes[escolhido.predict(novas)])
escolhido.predict_proba(novas).round(2)

**Para pensar:** em qual das duas o modelo ficou em dúvida? Olhe o gráfico de pétalas: onde essa flor cairia?

### 5.2 O que a floresta olhou?
A Random Forest conta quanto cada feature ajudou a separar as classes, somando todas as árvores.

In [ ]:
rf = modelos['Random Forest'].fit(X_tr, y_tr)
# TODO: importância de cada feature, da maior para a menor
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).round(3)

## Para discutir
1. O modelo que foi melhor **no treino** deve necessariamente ser o escolhido?
2. Por que escolher pela validação cruzada e só depois olhar o teste?
3. Onde o modelo errou na matriz de confusão? Essas espécies se parecem no gráfico de pétalas?

> 🔎 **Gabarito · o que observar**
>
> **Pontos para a discussão:**
> 1. Não. Ir bem no treino pode ser decoreba (K = 1, árvore sem limite). Escolhe-se pela validação cruzada.
> 2. Se o teste for usado para escolher, ele vira parte do treino e deixa de medir o desempenho em dados novos: a nota final sai otimista.
> 3. Todos os erros foram entre versicolor e virginica, que se sobrepõem no gráfico de pétalas. A setosa, isolada, nunca erra.